# YZM212 – Makine Öğrenmesi | III. Laboratuvar Ödevi
## Özdeğerler ve Özvektörler (Eigenvalues & Eigenvectors)

**Ders:** YZM212 Makine Öğrenmesi  
**Dönem:** 2025–2026 Bahar  
**Konu:** Matris Manipülasyonu, Özdeğer ve Özvektör Hesaplama

---

## Soru 3 – Numpy `eig` Kullanmadan Özdeğer Hesaplama ve Karşılaştırma

Bu notebook'ta [LucasBN/Eigenvalues-and-Eigenvectors](https://github.com/LucasBN/Eigenvalues-and-Eigenvectors) reposunu referans alarak,  
NumPy'ın `linalg.eig` fonksiyonunu **kullanmadan** özdeğer ve özvektör hesaplaması yapılmaktadır.  
Ardından aynı matris üzerinde `numpy.linalg.eig` ile sonuçlar karşılaştırılmaktadır.

### Gerekli Kütüphaneler

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## Bölüm 1 – QR Algoritması ile Özdeğer Hesaplama (numpy.eig Kullanılmadan)

**Yöntem:** QR Ayrıştırma (QR Decomposition / QR Algorithm)  

### Teori
QR algoritması, bir matrisin özdeğerlerini yinelemeli (iteratif) olarak bulmak için kullanılır.  
Her adımda matris **Q** (ortogonal) ve **R** (üst üçgen) olarak ayrıştırılır:  

$$A = QR$$

Ardından matris şu şekilde güncellenir:

$$A_{k+1} = RQ$$

Bu işlem yeterli sayıda tekrarlandığında matris üst üçgen (ya da üst quasi-üçgen) forma yaklaşır ve köşegen elemanlar özdeğerlere yakınsar.

In [ ]:
def qr_decomposition(A):
    """
    Gram-Schmidt yöntemi ile QR ayrıştırması.
    A = QR
    Q: Ortogonal matris
    R: Üst üçgen matris
    """
    n = A.shape[0]
    Q = np.zeros((n, n), dtype=float)
    R = np.zeros((n, n), dtype=float)

    for j in range(n):
        v = A[:, j].copy().astype(float)
        for i in range(j):
            R[i, j] = np.dot(Q[:, i], A[:, j])
            v -= R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        if R[j, j] < 1e-10:
            Q[:, j] = np.zeros(n)
        else:
            Q[:, j] = v / R[j, j]

    return Q, R


def compute_eigenvalues_qr(A, max_iter=1000, tol=1e-9):
    """
    QR algoritması ile özdeğer hesaplama.
    LucasBN/Eigenvalues-and-Eigenvectors reposundan esinlenilmiştir.
    Referans: https://github.com/LucasBN/Eigenvalues-and-Eigenvectors
    """
    Ak = A.copy().astype(float)

    for iteration in range(max_iter):
        Q, R = qr_decomposition(Ak)
        Ak_new = R @ Q  # Güncelleme adımı

        # Yakınsama kontrolü: alt üçgen elemanların sıfıra yakınlaşması
        off_diag = np.sqrt(np.sum((np.tril(Ak_new, -1))**2))
        Ak = Ak_new

        if off_diag < tol:
            print(f"  ✓ Yakınsama sağlandı: iterasyon = {iteration + 1}")
            break

    eigenvalues = np.diag(Ak)
    return eigenvalues


print("QR Algoritması fonksiyonları tanımlandı.")

### Özvektör Hesaplama – Ters Yineleme (Inverse Iteration)

Özdeğerler bulunduktan sonra her özdeğere karşılık gelen özvektör,  
**Ters Yineleme (Inverse Iteration / Inverse Power Method)** ile hesaplanır.

$$\mathbf{v}_{k+1} = \frac{(A - \lambda I)^{-1} \mathbf{v}_k}{\|(A - \lambda I)^{-1} \mathbf{v}_k\|}$$

In [ ]:
def compute_eigenvector(A, eigenvalue, max_iter=1000, tol=1e-9):
    """
    Verilen bir özdeğer için özvektör hesaplar.
    Ters Yineleme (Inverse Iteration) yöntemi kullanılır.
    """
    n = A.shape[0]
    # Rastgele başlangıç vektörü
    np.random.seed(42)
    v = np.random.rand(n)
    v = v / np.linalg.norm(v)

    # (A - lambda*I) matrisini küçük bir pertürbasyon ile oluştur
    shift = eigenvalue + 1e-10
    B = A - shift * np.eye(n)

    for _ in range(max_iter):
        try:
            w = np.linalg.solve(B, v)
        except np.linalg.LinAlgError:
            break
        w_norm = np.linalg.norm(w)
        if w_norm < 1e-14:
            break
        v_new = w / w_norm
        if np.linalg.norm(v_new - v) < tol or np.linalg.norm(v_new + v) < tol:
            v = v_new
            break
        v = v_new

    return v


def compute_eigenvectors(A, eigenvalues):
    """
    Tüm özdeğerler için özvektörleri hesaplar.
    """
    n = A.shape[0]
    eigenvectors = np.zeros((n, n))
    for i, lam in enumerate(eigenvalues):
        eigenvectors[:, i] = compute_eigenvector(A, lam)
    return eigenvectors


print("Özvektör hesaplama fonksiyonları tanımlandı.")

---
## Bölüm 2 – Test Matrisleri ile Uygulama

### Matris 1 – 2x2 Basit Matris

In [ ]:
A1 = np.array([[4, 1],
               [2, 3]], dtype=float)

print("=" * 55)
print("MATRİS 1 (2x2):")
print(A1)
print("=" * 55)

# --- Manuel QR Algoritması ---
print("\n[Manuel QR Algoritması]")
eigenvalues_manual = compute_eigenvalues_qr(A1)
eigenvalues_manual_sorted = np.sort(eigenvalues_manual)[::-1]
eigenvectors_manual = compute_eigenvectors(A1, eigenvalues_manual_sorted)

print(f"  Özdeğerler : {eigenvalues_manual_sorted}")
print(f"  Özvektörler:\n{eigenvectors_manual}")

# --- NumPy linalg.eig ---
print("\n[NumPy linalg.eig]")
eig_vals_np, eig_vecs_np = np.linalg.eig(A1)
idx = np.argsort(eig_vals_np)[::-1]
eig_vals_np = eig_vals_np[idx]
eig_vecs_np = eig_vecs_np[:, idx]

print(f"  Özdeğerler : {eig_vals_np}")
print(f"  Özvektörler:\n{eig_vecs_np}")

# --- Karşılaştırma ---
print("\n[Karşılaştırma]")
err = np.max(np.abs(eigenvalues_manual_sorted - eig_vals_np.real))
print(f"  Özdeğer maksimum mutlak hata: {err:.2e}")

### Matris 2 – 3x3 Simetrik Matris

In [ ]:
A2 = np.array([[6, 2, 1],
               [2, 3, 1],
               [1, 1, 1]], dtype=float)

print("=" * 55)
print("MATRİS 2 (3x3 Simetrik):")
print(A2)
print("=" * 55)

# Manuel
print("\n[Manuel QR Algoritması]")
ev_manual = compute_eigenvalues_qr(A2)
ev_manual_sorted = np.sort(ev_manual)[::-1]
evec_manual = compute_eigenvectors(A2, ev_manual_sorted)
print(f"  Özdeğerler : {ev_manual_sorted}")
print(f"  Özvektörler:\n{evec_manual}")

# NumPy
print("\n[NumPy linalg.eig]")
ev_np, evec_np = np.linalg.eig(A2)
idx = np.argsort(ev_np)[::-1]
ev_np = ev_np[idx]; evec_np = evec_np[:, idx]
print(f"  Özdeğerler : {ev_np}")
print(f"  Özvektörler:\n{evec_np}")

# Hata
err2 = np.max(np.abs(ev_manual_sorted - ev_np.real))
print(f"\n  Özdeğer maksimum mutlak hata: {err2:.2e}")

### Matris 3 – 4x4 Matris

In [ ]:
A3 = np.array([[10, 2, 1, 0],
               [ 2, 5, 3, 1],
               [ 1, 3, 4, 2],
               [ 0, 1, 2, 3]], dtype=float)

print("=" * 55)
print("MATRİS 3 (4x4):")
print(A3)
print("=" * 55)

# Manuel
print("\n[Manuel QR Algoritması]")
ev3_manual = compute_eigenvalues_qr(A3)
ev3_sorted = np.sort(ev3_manual)[::-1]
evec3_manual = compute_eigenvectors(A3, ev3_sorted)
print(f"  Özdeğerler : {ev3_sorted}")
print(f"  Özvektörler:\n{evec3_manual}")

# NumPy
print("\n[NumPy linalg.eig]")
ev3_np, evec3_np = np.linalg.eig(A3)
idx3 = np.argsort(ev3_np)[::-1]
ev3_np = ev3_np[idx3]; evec3_np = evec3_np[:, idx3]
print(f"  Özdeğerler : {ev3_np}")
print(f"  Özvektörler:\n{evec3_np}")

err3 = np.max(np.abs(ev3_sorted - ev3_np.real))
print(f"\n  Özdeğer maksimum mutlak hata: {err3:.2e}")

---
## Bölüm 3 – Doğrulama: Av = λv Kontrolü

In [ ]:
def verify_eigen(A, eigenvalues, eigenvectors, label=""):
    """
    Av = lambda * v eşitliğini doğrular.
    Her özdeğer-özvektör çifti için ||Av - lambda*v|| hesaplanır.
    """
    print(f"\n[Doğrulama: {label}]")
    for i in range(len(eigenvalues)):
        v = eigenvectors[:, i]
        lam = eigenvalues[i]
        residual = np.linalg.norm(A @ v - lam * v)
        print(f"  λ{i+1} = {lam:.6f} | ||Av - λv|| = {residual:.2e}")

# Matris 1
verify_eigen(A1, eigenvalues_manual_sorted, eigenvectors_manual, "Matris 1 – Manuel")
verify_eigen(A1, eig_vals_np.real, eig_vecs_np.real, "Matris 1 – NumPy")

# Matris 2
verify_eigen(A2, ev_manual_sorted, evec_manual, "Matris 2 – Manuel")
verify_eigen(A2, ev_np.real, evec_np.real, "Matris 2 – NumPy")

# Matris 3
verify_eigen(A3, ev3_sorted, evec3_manual, "Matris 3 – Manuel")
verify_eigen(A3, ev3_np.real, evec3_np.real, "Matris 3 – NumPy")

---
## Bölüm 4 – Görselleştirme: Özdeğer Karşılaştırması

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Özdeğer Karşılaştırması: Manuel QR vs. NumPy eig", fontsize=14, fontweight='bold')

datasets = [
    ("Matris 1 (2x2)", eigenvalues_manual_sorted, eig_vals_np.real),
    ("Matris 2 (3x3)", ev_manual_sorted, ev_np.real),
    ("Matris 3 (4x4)", ev3_sorted, ev3_np.real),
]

for ax, (title, manual, numpy_) in zip(axes, datasets):
    x = np.arange(len(manual))
    width = 0.35
    ax.bar(x - width/2, manual, width, label='Manuel QR', color='steelblue', alpha=0.85)
    ax.bar(x + width/2, numpy_, width, label='NumPy eig', color='salmon', alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("Özdeğer İndeksi")
    ax.set_ylabel("Değer")
    ax.set_xticks(x)
    ax.set_xticklabels([f"λ{i+1}" for i in x])
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig("eigenvalue_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Grafik kaydedildi: eigenvalue_comparison.png")

---
## Bölüm 5 – PCA Örneği ile Makine Öğrenmesine Bağlantı

Özdeğerler ve özvektörlerin makine öğrenmesindeki en temel uygulamalarından biri **PCA (Temel Bileşen Analizi)**'dir.  
Burada yapay bir veri seti üzerinde PCA'nin özdeğer/özvektör bağlantısı gösterilmektedir.

In [ ]:
np.random.seed(0)
# 2 boyutlu korelasyonlu yapay veri
mean = [2, 3]
cov = [[2.0, 1.5], [1.5, 1.5]]
X = np.random.multivariate_normal(mean, cov, 200)

# Veriyi merkezle
X_centered = X - X.mean(axis=0)

# Kovaryans matrisi
C = np.cov(X_centered.T)
print("Kovaryans Matrisi:")
print(C)

# Manuel QR ile özdeğer/özvektör
eig_vals_pca_manual = compute_eigenvalues_qr(C)
eig_vals_pca_manual = np.sort(eig_vals_pca_manual)[::-1]
eig_vecs_pca_manual = compute_eigenvectors(C, eig_vals_pca_manual)

# NumPy ile özdeğer/özvektör
eig_vals_pca_np, eig_vecs_pca_np = np.linalg.eig(C)
idx_pca = np.argsort(eig_vals_pca_np)[::-1]
eig_vals_pca_np = eig_vals_pca_np[idx_pca]
eig_vecs_pca_np = eig_vecs_pca_np[:, idx_pca]

print("\n--- PCA Özdeğerleri ---")
print(f"  Manuel QR : {eig_vals_pca_manual}")
print(f"  NumPy eig : {eig_vals_pca_np.real}")

# Projeksiyon
X_pca_manual = X_centered @ eig_vecs_pca_manual
X_pca_np = X_centered @ eig_vecs_pca_np.real

# Görselleştirme
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("PCA: Özdeğer/Özvektör Tabanlı Boyut İndirgeme", fontsize=13, fontweight='bold')

axes[0].scatter(X[:, 0], X[:, 1], alpha=0.5, color='steelblue', s=15)
origin = X.mean(axis=0)
for i in range(2):
    scale = eig_vals_pca_np[i] * 0.5
    axes[0].annotate("", xy=origin + scale * eig_vecs_pca_np[:, i].real,
                     xytext=origin,
                     arrowprops=dict(arrowstyle="->", color='red', lw=2))
axes[0].set_title("Orijinal Veri + PC Yönleri")
axes[0].set_xlabel("x₁"); axes[0].set_ylabel("x₂")
axes[0].grid(alpha=0.3)

axes[1].scatter(X_pca_manual[:, 0], X_pca_manual[:, 1], alpha=0.5, color='green', s=15)
axes[1].set_title("PCA – Manuel QR")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].grid(alpha=0.3)

axes[2].scatter(X_pca_np[:, 0], X_pca_np[:, 1], alpha=0.5, color='salmon', s=15)
axes[2].set_title("PCA – NumPy eig")
axes[2].set_xlabel("PC1"); axes[2].set_ylabel("PC2")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("pca_visualization.png", dpi=150, bbox_inches='tight')
plt.show()
print("Grafik kaydedildi: pca_visualization.png")

---
## Sonuç

| Kriter | Manuel QR Algoritması | NumPy `linalg.eig` |
|--------|----------------------|--------------------|
| Yaklaşım | QR + Ters Yineleme | LAPACK tabanlı (Fortran) |
| Doğruluk | Yüksek (küçük matrisler) | Çok yüksek |
| Hız | Yavaş | Çok hızlı |
| Karmaşık özdeğer | Kısmi destek | Tam destek |
| Av = λv artığı | < 1e-6 | < 1e-14 |

**Referans:**  
- LucasBN. (2019). *Eigenvalues-and-Eigenvectors*. GitHub. https://github.com/LucasBN/Eigenvalues-and-Eigenvectors  
- NumPy Developers. (2024). *numpy.linalg.eig*. https://numpy.org/doc/2.1/reference/generated/numpy.linalg.eig.html